> 在线讲义：[第 22 章 · Actor-Critic、优势函数与 GAE：演员演，评委打分](https://wikieden.github.io/rl-from-scratch/lessons/22-actor-critic-gae.html) · 本书《从零学强化学习》

# 第 22 章 · Actor-Critic、优势函数与 GAE：演员演，评委打分

本章回答：baseline 换成一个会学的价值网络，会得到什么算法？"优势"是什么意思？TD 误差怎么当成优势用？GAE 的 λ 在调什么？

配套讲义：仓库里 `cs285-deep-rl/notes/lessons/22-actor-critic-gae.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)  # 打印保留 3 位小数
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 代码 | 跑出来 |
|---|---|---|
| δ | `td_errors([0, 0, 1], [0.5, 0.6, 0.8, 0.0], 0.9)` | [0.04, 0.12, 0.2] |
| GAE λ = 0 / 0.95 / 1 | `gae(r, v, 0.9, lam)` | [0.04, 0.12, 0.2] / [0.289, 0.291, 0.2] / [0.31, 0.3, 0.2] |
| MC 优势 | `mc_advantage(r, v, 0.9)` | [0.31, 0.3, 0.2] |

## 1. 先把要用的东西定义好

这几格只是定义（环境、函数），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `td_errors`**（来自 `ch22_actor_critic_gae.py`）：values 比 rewards 多一个（最后一个是结束后的状态价值，结束时填 0）。

In [ ]:
def td_errors(rewards, values, gamma):
    """values 比 rewards 多一个（最后一个是结束后的状态价值，结束时填 0）。"""
    r, v = np.asarray(rewards, float), np.asarray(values, float)
    return r + gamma * v[1:] - v[:-1]

**函数 `gae`**（来自 `ch22_actor_critic_gae.py`）：下面要用到的小工具。

In [ ]:
def gae(rewards, values, gamma, lam):
    delta, A, out = td_errors(rewards, values, gamma), 0.0, []
    for d in reversed(delta):
        A = d + gamma * lam * A
        out.append(A)
    return np.array(out[::-1])

**函数 `mc_advantage`**（来自 `ch22_actor_critic_gae.py`）：下面要用到的小工具。

In [ ]:
def mc_advantage(rewards, values, gamma):
    G, out = 0.0, []
    for r in reversed(rewards):
        G = r + gamma * G
        out.append(G)
    return np.array(out[::-1]) - np.asarray(values[:-1], float)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
r, v, g = [0, 0, 1], [0.5, 0.6, 0.8, 0.0], 0.9

In [ ]:
print("δ =", np.round(td_errors(r, v, g), 3))

In [ ]:
for lam in [0.0, 0.95, 1.0]:
        print(f"λ = {lam}：Â =", np.round(gae(r, v, g, lam), 3))

In [ ]:
print("MC 优势 G − V =", np.round(mc_advantage(r, v, g), 3))

## 3. 代码里才看得到的

`values` 要比 `rewards` 多一个元素：最后一个状态之后的价值（结束了就是 0，没结束是被截断时评委的估计）。HW2 的 `_estimate_advantage` 里，这个"结束了填 0"由 `terminals` 数组控制，弄错会让最后一步的 δ 错得离谱。

## 4. 自己改一改

讲义里建议的几个小实验（数字是实跑出来的，改完对一对）：

1. 换 λ：`gae(r, v, 0.9, 0.5)` → [0.135, 0.21, 0.2]；λ = 0.9 → [0.268, 0.282, 0.2]。λ 从 0 往 1 走，优势从单步 δ 慢慢变成 MC 的结果。
2. γ 换成 0.99：δ 变成 [0.094, 0.192, 0.2]，GAE（λ = 0.95）变成 [0.451, 0.38, 0.2]。

讲义末尾的练习题也可以在下面的空格子里用代码验算：先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 优势 = Q − V：比平常好多少，按策略平均为零；
> 演员做动作，评委打分；TD 误差 δ 就是优势的估计；
> GAE：后面的 δ 按 γλ 打折相加；λ = 0 是一步 TD，λ = 1 是 MC；
> 跑一批、训评委、算 GAE、归一化、演员走一步：PPO 的前一半。